# LayoutLM Document QA (impira) — DIMER extractive document question answering tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/layoutlm-document-qa-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/layoutlm-document-qa-pipeline/blob/main/tutorials/layoutlm_document_qa_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-impira%2Flayoutlm--document--qa-ffcc4d?style=flat)](https://huggingface.co/impira/layoutlm-document-qa) [![Upstream](https://img.shields.io/badge/Upstream-microsoft%2Funilm%20(layoutlm)-181717?style=flat&logo=github&logoColor=white)](https://github.com/microsoft/unilm/tree/master/layoutlm) [![arXiv](https://img.shields.io/badge/arXiv-1912.13318-b31b1b.svg)](https://arxiv.org/abs/1912.13318)

**Profile:** `TASK-INFERENCE`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** extractive document question answering — one question plus the page's OCR words and pixel boxes → the best answer span with its start/end word indices and span score — using the pinned `impira/layoutlm-document-qa` weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/layoutlm_document_qa_pipeline/pipeline.py` at revision `7cf5358c1da0`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `beed3c4d02d86017ebca5bd0fdf210046b907aa6` (~514 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned snapshot, obtains the tutorial sample automatically, validates it into an input manifest before the model runs, runs the task locally in this kernel, writes the evaluation report, and exports machine-readable outputs with provenance. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5).

**Bring Your Own Data:** After the sample workflow completes, set `USE_BYOD = True` in the sample cell and re-run from that cell to supply your own input. It passes through the same notebook-local validation, task, evaluation-report and export cells as the sample; the expected input format, the ceilings and the privacy guidance are stated in the Prerequisites and in the sample cell, and the upload stays inside this runtime. BYOD is optional and never part of the default path.

At inference the LayoutLM (v1) encoder — a 12-layer BERT-style transformer with 2-D position embeddings, 128M parameters, RoBERTa vocabulary, fine-tuned by Impira on SQuAD 2.0 and DocVQA — reads the question and the page's **words with their boxes** (normalised to a 0–1000 grid) as one token sequence and predicts a start and an end position; the pipeline scores every span of at most 15 tokens by the product of the start and end softmax probabilities and returns the best one as a word span. **The model never sees pixels:** OCR is an input provider outside the model, so the pipeline takes words and boxes from the caller (bring-your-own OCR) and offers an optional Tesseract adapter that is imported only when called. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting happens in this notebook — the upstream checkpoint supplies the weights and tokenizer, and the carried module adds snapshot verification, the input contract (words, one pixel box per word inside the page, a non-empty question up to 256 characters), windowing for long pages, a fixed output contract, and the `anls`, `exact_match`, `validate_inputs` and `evaluation_report` helpers. The default sample is an invoice-style form rendered in code whose renderer records every word's box, with five authored questions and accepted answers, so ANLS and exact-match are demonstration (plumbing) evidence for one page with perfect OCR, not a DocVQA benchmark.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, resolve and digest-verify the immutable upstream model revision, render a synthetic form whose words and boxes are known exactly (or bring your own page with your own OCR output and questions) and validate it into an input manifest, run the supported task, read the answers correctly (a word span, its indices, an uncalibrated span score), exercise an optional BYOD path, produce an evaluation report that is `sample-sanity` with `anls` and `exact_match` only when accepted answers exist and `not-measurable` otherwise, and export the answers, the annotated page with the answer boxes and provenance.

**This notebook does not demonstrate:** OCR itself (the notebook installs no Tesseract and ships no OCR model; the default path uses the renderer's own word boxes and BYOD needs either `pytesseract` with a Tesseract binary already present or a JSON of words and boxes from your own OCR), PDF or multi-page documents (one page per call), abstention (the model always returns its best span, even for an unanswerable question), answers that are not a contiguous span of the OCR words, arithmetic or reasoning, evaluation on the DocVQA benchmark (registration-gated and not bundled), and any training. Answer quality is bounded by the OCR: a mis-read or mis-ordered word cannot be recovered by the model.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; inference is float32 on both. CPU is more than adequate: the repository's model card records 5.1 s to load and 0.11–0.29 s per question on the 73-word rendered form in the Windows venv (Intel Core Ultra 9 275HX). The pinned `torch==2.14.0` install and the 511 MB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python and PIL; what extractive (span) question answering is; what a start/end softmax product means and why it is not calibrated; what normalised Levenshtein similarity (ANLS) measures.
- **Data:** the default sample is a deterministic 850×1100 invoice-style form rendered in code with Pillow's bundled font, whose renderer records the pixel box of every word it draws (73 words) — perfect OCR by construction — with five authored questions and accepted answers, so nothing is downloaded and no private data is needed. Optional BYOD is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one page image decodable by Pillow plus **either** a `pytesseract`-importable Tesseract installation in the runtime **or** a JSON file `{"words": [...], "boxes": [[x0, y0, x1, y1], ...]}` in the page's pixel coordinates from your own OCR, plus your questions typed into the form field. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `impira/layoutlm-document-qa` snapshot (~514 MB in total) at revision `beed3c4d02d8…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'layoutlm-document-qa-pipeline',
    'repository_revision': '7cf5358c1da0ef63a4fc61aac3330d100cf9f0b6',
    'embedded_module': 'src/layoutlm_document_qa_pipeline/pipeline.py',
    'embedded_modules': ['src/layoutlm_document_qa_pipeline/pipeline.py'],
    'module_sha256': '66268e5891c2bb3178302b0abb9fb4c3ed1e955e5f0f23c77d6d3b042e0c30aa',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/layoutlm_document_qa_pipeline/` @ `7cf5358c1da0`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/layoutlm_document_qa_pipeline/pipeline.py`

In [ ]:
"""Extractive document question answering with the pinned ``impira/layoutlm-document-qa`` checkpoint.

The class loads the tokenizer and model only from a digest-verified local snapshot (``weights/<key>/``)
or, when explicitly allowed, from the Hugging Face Hub at the pinned revision — always with
``trust_remote_code=False``: the LayoutLM architecture comes from the pinned ``transformers`` release,
the weights are SafeTensors, and no model-repository code is executed.

LayoutLM (v1) reads **words and their boxes**, not pixels: OCR is an input provider outside the model.
The pipeline therefore takes ``words``/``boxes`` from the caller (bring-your-own OCR) and offers
``ocr_words_with_tesseract`` as an optional adapter that is imported only when called.
"""

from __future__ import annotations

import hashlib
import json
import re
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from PIL import Image

MODEL_ID = "impira/layoutlm-document-qa"
MODEL_REVISION = "beed3c4d02d86017ebca5bd0fdf210046b907aa6"
MODEL_LICENSE = "mit"
MODEL_KEY = "layoutlm-document-qa"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Encoding ceilings. The checkpoint's max_position_embeddings is 514 (RoBERTa layout: 512 usable
# tokens); longer documents are split into overlapping windows of MAX_SEQ_LEN with DOC_STRIDE overlap
# and the best-scoring span across windows is returned (the transformers document-question-answering
# pipeline's convention, as is MAX_ANSWER_TOKENS).
MAX_SEQ_LEN = 512
DOC_STRIDE = 128
MAX_ANSWER_TOKENS = 15
MAX_WORDS = 2000
MAX_QUESTION_CHARS = 256
# Box ceilings. Boxes are pixel xyxy in the page's coordinate frame and are normalised to the
# 0..1000 grid LayoutLM expects (max_2d_position_embeddings 1024).
MAX_IMAGE_SIDE = 10000
MIN_IMAGE_SIDE = 1
BOX_GRID = 1000
# ANLS (DocVQA's official metric): a normalised Levenshtein similarity below this threshold scores 0.
ANLS_THRESHOLD = 0.5
_PUNCT_RE = re.compile(r"[^\w\s]")


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def normalize_answer(text: str) -> str:
    """DocVQA-style normalisation: lower-case, punctuation removed, whitespace collapsed."""
    return " ".join(_PUNCT_RE.sub(" ", text.lower()).split())


def _levenshtein(a: str, b: str) -> int:
    previous = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        current = [i]
        for j, cb in enumerate(b, 1):
            current.append(min(current[-1] + 1, previous[j] + 1, previous[j - 1] + (ca != cb)))
        previous = current
    return previous[-1]


def anls(prediction: str, golds: Sequence[str], *, threshold: float = ANLS_THRESHOLD) -> float:
    """Average Normalised Levenshtein Similarity for one question (Biten et al., ICDAR 2019).

    ``1 - lev(pred, gold) / max(len(pred), len(gold))`` over normalised strings, maximised over the
    accepted ``golds``; a similarity below ``threshold`` scores 0 so a near-miss is not rewarded.
    """
    if not golds:
        raise ValueError("golds must contain at least one accepted answer")
    pred = normalize_answer(prediction)
    best = 0.0
    for gold in golds:
        ref = normalize_answer(gold)
        longest = max(len(pred), len(ref))
        similarity = 1.0 if longest == 0 else 1.0 - _levenshtein(pred, ref) / longest
        best = max(best, similarity)
    return best if best >= threshold else 0.0


def exact_match(prediction: str, golds: Sequence[str]) -> bool:
    """Whether the normalised prediction equals any normalised accepted answer."""
    pred = normalize_answer(prediction)
    return any(pred == normalize_answer(gold) for gold in golds)


def normalize_box(box: Sequence[float], width: int, height: int) -> list[int]:
    """Pixel xyxy -> LayoutLM's 0..1000 grid (the transformers pipeline's normalize_bbox)."""
    x0, y0, x1, y1 = (float(v) for v in box)
    return [
        int(BOX_GRID * (x0 / width)),
        int(BOX_GRID * (y0 / height)),
        int(BOX_GRID * (x1 / width)),
        int(BOX_GRID * (y1 / height)),
    ]


def ocr_words_with_tesseract(image: Image.Image, *, lang: str = "eng") -> tuple[list[str], list[list[float]]]:
    """Optional OCR adapter: words and pixel xyxy boxes from Tesseract via ``pytesseract``.

    Neither ``pytesseract`` nor the Tesseract binary is part of this package's pinned runtime; the
    import happens here so the rest of the pipeline stays usable with any OCR the caller prefers.
    """
    try:
        import pytesseract
    except ImportError as exc:  # pragma: no cover - depends on the host
        raise RuntimeError("pytesseract is not installed; supply words and boxes from your own OCR") from exc
    data = pytesseract.image_to_data(image.convert("RGB"), lang=lang, output_type=pytesseract.Output.DICT)
    words: list[str] = []
    boxes: list[list[float]] = []
    for text, left, top, w, h in zip(
        data["text"], data["left"], data["top"], data["width"], data["height"], strict=True
    ):
        token = str(text).strip()
        if not token:
            continue
        words.append(token)
        boxes.append([float(left), float(top), float(left + w), float(top + h)])
    return words, boxes


INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        "one question string plus the page's OCR words (list of str) with one pixel xyxy box per word "
        "and the page size (width, height) the boxes are expressed in; pixels are not read by the model"
    ),
    "words": [1, MAX_WORDS],
    "question_chars": [1, MAX_QUESTION_CHARS],
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "encoding": (
        f"<s> question </s></s> words </s> with RoBERTa byte-level BPE; boxes normalised to 0..{BOX_GRID}; "
        f"windows of {MAX_SEQ_LEN} tokens with {DOC_STRIDE}-token overlap when the words do not fit"
    ),
    "output": (
        f"the best word span of at most {MAX_ANSWER_TOKENS} tokens with its start/end word indices and the "
        "span score (softmax start x softmax end within the window)"
    ),
}


def _check_page(image_size: Any) -> tuple[int, int]:
    if (
        not isinstance(image_size, Sequence)
        or isinstance(image_size, str)
        or len(image_size) != 2
        or any(isinstance(v, bool) or not isinstance(v, int) for v in image_size)
    ):
        raise TypeError("image_size must be a (width, height) pair of ints")
    width, height = int(image_size[0]), int(image_size[1])
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return width, height


def _check_document(
    words: Any, boxes: Any, image_size: Any
) -> tuple[list[str], list[list[int]], tuple[int, int]]:
    """Raise TypeError/ValueError naming the first violated ceiling; return words, grid boxes, size."""
    width, height = _check_page(image_size)
    if isinstance(words, str) or not isinstance(words, Sequence):
        raise TypeError("words must be a sequence of str")
    if not 1 <= len(words) <= MAX_WORDS:
        raise ValueError(f"words has {len(words)} entries; expected 1..MAX_WORDS={MAX_WORDS}")
    if not all(isinstance(word, str) and word.strip() for word in words):
        raise ValueError("every word must be a non-empty str")
    if isinstance(boxes, str) or not isinstance(boxes, Sequence) or len(boxes) != len(words):
        raise ValueError(f"boxes must have one pixel xyxy box per word ({len(words)})")
    grid: list[list[int]] = []
    for index, box in enumerate(boxes):
        if isinstance(box, str) or not isinstance(box, Sequence) or len(box) != 4:
            raise ValueError(f"boxes[{index}] must be [x0, y0, x1, y1]")
        try:
            x0, y0, x1, y1 = (float(v) for v in box)
        except (TypeError, ValueError) as exc:
            raise ValueError(f"boxes[{index}] must hold numbers") from exc
        if not (0 <= x0 <= x1 <= width and 0 <= y0 <= y1 <= height):
            raise ValueError(f"boxes[{index}] {list(box)} is not inside the {width}x{height} page")
        grid.append(normalize_box((x0, y0, x1, y1), width, height))
    return [str(word) for word in words], grid, (width, height)


def _check_question(question: Any) -> str:
    if not isinstance(question, str):
        raise TypeError("question must be a str")
    checked = " ".join(question.split())
    if not checked:
        raise ValueError("question must contain at least one non-whitespace character")
    if len(checked) > MAX_QUESTION_CHARS:
        raise ValueError(f"question has {len(checked)} chars > MAX_QUESTION_CHARS {MAX_QUESTION_CHARS}")
    return checked


def validate_inputs(
    words: Sequence[str],
    boxes: Sequence[Sequence[float]],
    questions: Sequence[str],
    *,
    image_size: Sequence[int],
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Every question is checked exactly as ``answer`` would check it; rejection is reported by raising,
    and a caller that wants the finding recorded catches the exception and stores ``str(exc)`` under
    ``findings``.
    """
    checked_words, _grid, size = _check_document(words, boxes, image_size)
    if isinstance(questions, str) or not isinstance(questions, Sequence) or not questions:
        raise TypeError("questions must be a non-empty sequence of str")
    checked = [_check_question(question) for question in questions]
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (answer takes one page)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {"id": names[0] if names else "page-0", "size": list(size), "n_words": len(checked_words)}
        ],
        "questions": checked,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    results: Sequence[Mapping[str, Any]],
    golds: Sequence[Sequence[str]] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``golds`` (one sequence of accepted answers per result, in order) the report carries the
    mean ``anls`` and the ``exact_match`` rate over the questions plus one per-question entry, verdict
    ``sample-sanity``; without golds it is ``not-measurable`` and says what labelled data would make
    the task measurable.
    """
    if not results:
        raise ValueError("results must contain at least one answer result")
    base = {
        "task": "question + OCR words/boxes -> extractive answer span (LayoutLM v1)",
        "score_semantics": (
            "score is the product of the start and end softmax probabilities of the chosen span within "
            "its window under the model's own head — a ranking signal over spans of this page, not a "
            "calibrated probability that the answer is right, and never a signal that the question is "
            "answerable; the model always returns its best span"
        ),
        "sample_kind": sample_kind,
        "n_questions": len(results),
        "scores": [float(result.get("score", 0.0)) for result in results],
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if golds is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no accepted answers were supplied for the evaluated questions",
            "needs": (
                "question/answer pairs with accepted answers on pages from the deployment domain "
                "(DocVQA-style annotations) scored with ANLS, plus the OCR the deployment will really use; "
                "no such labelled set ships with this repository"
            ),
        }
    if len(golds) != len(results):
        raise ValueError(f"golds has {len(golds)} entries for {len(results)} results")
    per_question = []
    for result, accepted in zip(results, golds, strict=True):
        if isinstance(accepted, str) or not accepted:
            raise ValueError("each golds entry must be a non-empty sequence of accepted answers")
        prediction = str(result["answer"])
        per_question.append(
            {
                "question": result.get("question"),
                "prediction": prediction,
                "score": float(result.get("score", 0.0)),
                "golds": list(accepted),
                "anls": anls(prediction, accepted),
                "exact_match": exact_match(prediction, accepted),
            }
        )
    metrics = [
        {
            "id": "anls",
            "value": sum(entry["anls"] for entry in per_question) / len(per_question),
            "threshold": ANLS_THRESHOLD,
            "normalisation": "lower-cased, punctuation removed, whitespace collapsed; max over golds",
            "estimation": f"{len(per_question)} question(s) on one page, no dispersion estimate",
        },
        {
            "id": "exact_match",
            "value": sum(entry["exact_match"] for entry in per_question) / len(per_question),
            "normalisation": "lower-cased, punctuation removed, whitespace collapsed",
            "estimation": f"{len(per_question)} question(s) on one page, no dispersion estimate",
        },
    ]
    return {
        **base,
        "metrics": metrics,
        "per_question": per_question,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(per_question)} authored question(s) on one tutorial page whose words and boxes you "
            "rendered yourself; plumbing evidence, not a DocVQA benchmark"
        ),
        "needs": (
            "a labelled question/answer set on pages from the deployment domain with the deployment's own "
            "OCR for any accuracy claim; the DocVQA benchmark itself is registration-gated and not bundled"
        ),
    }


@dataclass
class LayoutLMDocumentQAPipeline:
    """``_runner(question, words, grid_boxes)`` returns ``{"start": int, "end": int, "score": float}``
    (word indices, inclusive) or ``{"start": None, ...}`` when no span could be selected."""

    _runner: Callable[..., dict[str, Any]]
    device: str = "cpu"
    dtype: str = "float32"
    source: str = "injected"

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> LayoutLMDocumentQAPipeline:
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        common: dict[str, Any] = {"trust_remote_code": False}
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            location, common["local_files_only"], source = str(root), True, "local-snapshot"
        elif allow_download:
            location, common["revision"], source = MODEL_ID, MODEL_REVISION, "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import AutoTokenizer, LayoutLMForQuestionAnswering

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        tokenizer = AutoTokenizer.from_pretrained(location, **common)
        if not tokenizer.is_fast:
            raise RuntimeError("a fast tokenizer is required for word_ids/sequence_ids; snapshot has none")
        model = LayoutLMForQuestionAnswering.from_pretrained(location, dtype=torch.float32, **common)
        model = model.eval().to(resolved_device)
        sep_id = tokenizer.sep_token_id

        def runner(question: str, words: list[str], grid_boxes: list[list[int]]) -> dict[str, Any]:
            encoding = tokenizer(
                text=question.split(),
                text_pair=words,
                is_split_into_words=True,
                max_length=MAX_SEQ_LEN,
                stride=DOC_STRIDE,
                truncation="only_second",
                return_overflowing_tokens=True,
                return_token_type_ids=True,
                padding="max_length",
                return_tensors="pt",
            )
            n_windows = int(encoding["input_ids"].shape[0])
            best: dict[str, Any] = {"start": None, "end": None, "score": 0.0, "n_windows": n_windows}
            for window in range(n_windows):
                sequence_ids = encoding.sequence_ids(window)
                word_ids = encoding.word_ids(window)
                input_ids = encoding["input_ids"][window]
                bbox = []
                for input_id, sequence_id, word_id in zip(
                    input_ids.tolist(), sequence_ids, word_ids, strict=True
                ):
                    if sequence_id == 1:
                        bbox.append(grid_boxes[word_id])
                    elif input_id == sep_id:
                        bbox.append([BOX_GRID] * 4)
                    else:
                        bbox.append([0] * 4)
                inputs = {
                    "input_ids": input_ids.unsqueeze(0).to(resolved_device),
                    "attention_mask": encoding["attention_mask"][window].unsqueeze(0).to(resolved_device),
                    "token_type_ids": encoding["token_type_ids"][window].unsqueeze(0).to(resolved_device),
                    "bbox": torch.tensor(bbox).unsqueeze(0).to(resolved_device),
                }
                with torch.inference_mode():
                    outputs = model(**inputs)
                # Only document tokens may start or end an answer (the pipeline's p_mask).
                allowed = torch.tensor([sid == 1 for sid in sequence_ids], device=resolved_device)
                start = outputs.start_logits[0].float().masked_fill(~allowed, float("-inf")).softmax(-1)
                end = outputs.end_logits[0].float().masked_fill(~allowed, float("-inf")).softmax(-1)
                candidates = start[:, None] * end[None, :]
                candidates = torch.triu(candidates) - torch.triu(candidates, diagonal=MAX_ANSWER_TOKENS)
                flat = int(candidates.argmax())
                s_index, e_index = divmod(flat, candidates.shape[1])
                score = float(candidates[s_index, e_index])
                if score > best["score"] and word_ids[s_index] is not None and word_ids[e_index] is not None:
                    best = {
                        "start": word_ids[s_index],
                        "end": word_ids[e_index],
                        "score": score,
                        "n_windows": n_windows,
                    }
            return best

        return cls(runner, resolved_device, "float32", source)

    def answer(
        self,
        question: str,
        *,
        words: Sequence[str],
        boxes: Sequence[Sequence[float]],
        image_size: Sequence[int],
    ) -> dict[str, Any]:
        """Answer one question from the page's words and boxes; ``answer`` is the selected word span."""
        checked_words, grid, size = _check_document(words, boxes, image_size)
        checked_question = _check_question(question)
        raw = self._runner(checked_question, checked_words, grid)
        if not isinstance(raw, dict) or "start" not in raw or "end" not in raw or "score" not in raw:
            raise RuntimeError("runner must return a dict with 'start', 'end' and 'score'")
        start, end = raw["start"], raw["end"]
        if start is not None and not (0 <= int(start) <= int(end) < len(checked_words)):
            raise RuntimeError(f"runner returned an invalid word span {start}..{end}")
        span_words = checked_words[start : end + 1] if start is not None else []
        return {
            "answer": " ".join(span_words),
            "score": float(raw["score"]),
            "start": None if start is None else int(start),
            "end": None if end is None else int(end),
            "question": checked_question,
            "n_words": len(checked_words),
            "n_windows": int(raw.get("n_windows", 1)),
            "image_size": list(size),
            "device": self.device,
            "dtype": self.dtype,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `8`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `beed3c4d02d8…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `LayoutLMDocumentQAPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "layoutlm-document-qa",
  "modelId": "impira/layoutlm-document-qa",
  "revision": "beed3c4d02d86017ebca5bd0fdf210046b907aa6",
  "files": [
    {
      "path": "README.md",
      "bytes": 2326,
      "sha256": "40fd65fc734cc7eb73cc815465b8073bc4e5b7484406e537c7b8d763f88493f7"
    },
    {
      "path": "config.json",
      "bytes": 789,
      "sha256": "6d0fc068193109d0d053fa4de00963778beffbde05067c3e9f3454235044380f"
    },
    {
      "path": "merges.txt",
      "bytes": 456356,
      "sha256": "fe36cab26d4f4421ed725e10a2e9ddb7f799449c603a96e7f29b5a3c82a95862"
    },
    {
      "path": "model.safetensors",
      "bytes": 511200628,
      "sha256": "e4bbad3e4a1b5ae50c787b7afd6049a0bfa99fd823b50436e444e092ae2347b9"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 239,
      "sha256": "378eb3bf733eb16e65792d7e3fda5b8a4631387ca04d2015199c4d4f22ae554d"
    },
    {
      "path": "tokenizer.json",
      "bytes": 1355881,
      "sha256": "33465117406b9007673e8ba283f7f1383d9b5094df947481af60eec94ed7d7bd"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 315,
      "sha256": "ea11996be5d083d63c72700810b18a6cfdf78c55131b754a75ae94e66b0ad6ab"
    },
    {
      "path": "vocab.json",
      "bytes": 798293,
      "sha256": "ed19656ea1707df69134c4af35c8ceda2cc9860bf2c3495026153a133670ab5e"
    }
  ],
  "totalBytes": 513814827
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = LayoutLMDocumentQAPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Render the synthetic form (words + boxes) or optional BYOD

The default sample is **synthetic** and carries its own references: an invoice-style form — an `INVOICE` header, a supplier name and address, five labelled fields, a four-row line-item table, subtotal/VAT/total lines and a payment-terms sentence — is rendered with Pillow's bundled font at 850×1100, and the renderer records the pixel box of **every word it draws** (`ImageDraw.textbbox`), which is exactly the `words`/`boxes` input the model needs. That is perfect OCR by construction — real OCR mis-reads, splits and re-orders words, and nothing here measures that. Five questions are authored against the page with their accepted answers; they are the references for the `anls` and `exact_match` sanity checks later, not a labelled dataset. The image digest is printed for the record.

BYOD is optional and disabled by default. When enabled, upload one page image; the notebook then tries the carried `ocr_words_with_tesseract` adapter (which needs `pytesseract` and a Tesseract binary **already present** in the runtime — the notebook installs neither) and, failing that, asks you to upload a JSON of words and pixel boxes from your own OCR. Type your questions one per line. No accepted answers exist for them, so the evaluation report will be `not-measurable`. Nothing is validated in this cell — the next section hands the words, boxes and questions to the pipeline's own validation stage, which is the only checker.

In [ ]:
import hashlib
import io
import json

import numpy as np
from PIL import Image, ImageDraw, ImageFont

USE_BYOD = False  # @param {type:"boolean"}
byod_questions = 'What is the invoice number?\nWho is the customer?'  # @param {type:"string"}


def synthetic_form(width=850, height=1100):
    """Invoice-style form rendered with Pillow's bundled font; every drawn word is recorded with its pixel box."""
    page = Image.new('RGB', (width, height), 'white')
    d = ImageDraw.Draw(page)
    body, bold, head = ImageFont.load_default(size=18), ImageFont.load_default(size=20), ImageFont.load_default(size=30)
    words, boxes = [], []

    def put(x, y, text, font, fill='black'):
        for token in text.split():
            x0, y0, x1, y1 = d.textbbox((x, y), token, font=font)
            d.text((x, y), token, fill=fill, font=font)
            words.append(token)
            boxes.append([float(x0), float(y0), float(x1), float(y1)])
            x = x1 + d.textlength(' ', font=font)

    put(70, 60, 'INVOICE', head)
    put(70, 110, 'Northwind Traders Ltd.', bold)
    put(70, 136, '14 Harbour Road, Portsmouth PO1 3AX', body, (40, 40, 40))
    y = 200
    for label, value in [('Invoice number:', 'NW-2026-0417'), ('Invoice date:', '12 March 2026'), ('Due date:', '11 April 2026'), ('Customer:', 'Blue Yonder Airlines'), ('Purchase order:', 'PO-88213')]:
        put(70, y, label, bold)
        put(300, y, value, body)
        y += 32
    d.rectangle([70, 400, 780, 640], outline='black', width=2)
    cols = [70, 420, 540, 660, 780]
    rows = [('Description', 'Qty', 'Unit price', 'Amount'), ('Cargo pallets (standard)', '40', '$18.50', '$740.00'), ('Shrink wrap rolls', '12', '$9.25', '$111.00'), ('Handling fee', '1', '$65.00', '$65.00')]
    for r, row in enumerate(rows):
        yy = 400 + r * 48
        if r:
            d.line([(70, yy), (780, yy)], fill=(120, 120, 120), width=1)
        for c, cell in enumerate(row):
            put(cols[c] + 10, yy + 14, cell, bold if r == 0 else body)
    for c in cols[1:-1]:
        d.line([(c, 400), (c, 640)], fill=(120, 120, 120), width=1)
    put(540, 670, 'Subtotal:', bold)
    put(680, 670, '$916.00', body)
    put(540, 700, 'VAT (20%):', bold)
    put(680, 700, '$183.20', body)
    put(540, 736, 'Total due:', head)
    put(680, 736, '$1,099.20', head)
    put(70, 900, 'Payment terms: 30 days from invoice date. Bank: Solent Mutual, sort code 40-11-22.', body, (40, 40, 40))
    qa = [
        ('What is the invoice number?', ['NW-2026-0417']),
        ('Who is the customer?', ['Blue Yonder Airlines']),
        ('What is the total due?', ['$1,099.20', '1,099.20']),
        ('What is the due date?', ['11 April 2026']),
        ('How many cargo pallets were invoiced?', ['40']),
    ]
    return page, words, boxes, qa


if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    image_name = next(iter(uploaded))
    image = Image.open(io.BytesIO(uploaded[image_name]))
    image.load()
    try:
        words, boxes = ocr_words_with_tesseract(image)
        ocr_source = 'tesseract (pytesseract present in this runtime)'
    except RuntimeError as exc:
        print(f'{exc} -- upload a JSON file with "words" and pixel "boxes" from your own OCR')
        ocr_upload = files.upload()
        ocr_payload = json.loads(next(iter(ocr_upload.values())).decode('utf-8'))
        words, boxes = list(ocr_payload['words']), [list(b) for b in ocr_payload['boxes']]
        ocr_source = 'caller-supplied JSON'
    questions = [line.strip() for line in byod_questions.splitlines() if line.strip()]
    golds = None
    sample_kind = 'BYOD'
else:
    # Deterministic synthetic form: no randomness, so no seed is needed and the digest is stable per Pillow build.
    image, words, boxes, qa = synthetic_form()
    questions, golds = [q for q, _ in qa], [g for _, g in qa]
    image_name = 'synthetic_invoice_850x1100.png'
    ocr_source = 'renderer word boxes (perfect OCR by construction)'
    sample_kind = 'synthetic'

image_sha256 = hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest()
print({'sample_kind': sample_kind, 'name': image_name, 'size': image.size, 'rgb_sha256': image_sha256, 'n_words': len(words), 'ocr_source': ocr_source, 'n_questions': len(questions), 'has_golds': golds is not None})
print(' '.join(words[:24]) + ' …')

## 5. Validate the request → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `answer` applies — a page size within `MIN_IMAGE_SIDE`..`MAX_IMAGE_SIDE`, 1..`MAX_WORDS` non-empty words with exactly one pixel box each lying inside the page, and each question a non-empty string of at most `MAX_QUESTION_CHARS` characters (whitespace collapsed) — and returns an **input manifest** naming the schema (including the token encoding, the windowing and the span-scoring rule), the page size and word count, the checked questions and the verdict. The manifest is written to `outputs/layoutlm_document_qa_input_manifest.json`. To show what rejection looks like, the cell also validates a box that lies outside the page and records the pipeline's own error message as a finding. Inside the pipeline each box is normalised to LayoutLM's 0–1000 grid; the question and the words are tokenised together and, when they exceed 512 tokens, split into overlapping windows. The pipeline cannot tell whether the OCR is right or whether the question is answerable: that contract is the caller's.

In [ ]:
import json
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_WORDS': MAX_WORDS, 'MAX_QUESTION_CHARS': MAX_QUESTION_CHARS, 'MAX_SEQ_LEN': MAX_SEQ_LEN, 'DOC_STRIDE': DOC_STRIDE, 'MAX_ANSWER_TOKENS': MAX_ANSWER_TOKENS, 'BOX_GRID': BOX_GRID}})
input_manifest = validate_inputs(words, boxes, questions, image_size=image.size, names=[image_name])
# Demonstrate rejection on a request that breaks the contract; the finding is recorded, not swallowed.
try:
    validate_inputs(words[:1], [[0, 0, image.width + 1, 10]], questions, image_size=image.size)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'box-outside-page-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/layoutlm_document_qa_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 6. Answer the questions and read the scores correctly

`answer` returns, per question, a dict with `answer` (the selected word span joined by spaces), `start` and `end` (inclusive word indices into your `words`, so the span's boxes are known), `score`, `n_words`, `n_windows` (how many 512-token windows the page needed), the checked `question` and the model identity. The `score` is the **product of the start and end softmax probabilities of the chosen span within its window under the model's own head** — a ranking signal over spans of this page, not a calibrated probability that the answer is right, and never a signal that the question is answerable: the model always returns its best span. As recorded in the model card, the repository's CPU smoke on this same form answered all five authored questions exactly with scores of 0.999–1.000 — and answered `14 Harbour Road,` at 0.28 to "What is the delivery address?", a question the page cannot answer. The lower score is suggestive, not a rule; a deployment that wants to reject answers must choose its own threshold on its own labelled pages. Inference is deterministic on a fixed device and dtype (no sampling); CUDA kernel selection can move scores in the third or fourth decimal place.

In [ ]:
import time

results, seconds = [], []
for question in questions:
    t0 = time.time()
    results.append(pipe.answer(question, words=words, boxes=boxes, image_size=image.size))
    seconds.append(round(time.time() - t0, 2))
print({'device': pipe.device, 'seconds_per_question': seconds, 'n_windows': results[0]['n_windows']})
for result in results:
    print(f"Q: {result['question']}\n   A: {result['answer']!r}  score {result['score']:.4f}  words {result['start']}..{result['end']}")

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report. No accuracy is reported by default: DocVQA-style accuracy needs labelled question/answer pairs on pages from the deployment domain **with the deployment's own OCR**, and this repository ships none (the DocVQA benchmark itself is registration-gated). The repository's metric helpers are `anls` — Average Normalised Levenshtein Similarity, the benchmark's official metric: `1 − edits / max(len)` over normalised strings, maximised over the accepted answers, scored 0 below the 0.5 threshold — and `exact_match` after the same normalisation (lower-case, punctuation removed, whitespace collapsed). When accepted answers are supplied the report carries the mean `anls`, the `exact_match` rate and one entry per question (with its span score), verdict `sample-sanity`. On the synthetic path those answers are values **you rendered yourself** and the OCR is perfect by construction, so a perfect score proves only that the input contract, encoding, forward pass and span decoding round-trip. On BYOD no accepted answers exist, the verdict is `not-measurable`, and the report states what would make the task measurable. The report is written to `outputs/layoutlm_document_qa_evaluation_report.json`.

In [ ]:
report = evaluation_report(results, golds, sample_kind=sample_kind)
with open('outputs/layoutlm_document_qa_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps({k: v for k, v in report.items() if k not in ('metrics', 'per_question')}, indent=2))
for metric in report['metrics']:
    print(f"{metric['id']:12} {metric['value']:.3f}  ({metric['estimation']})")
for entry in report.get('per_question', []):
    print(f"  anls {entry['anls']:.2f}  exact {str(entry['exact_match']):5}  score {entry['score']:.3f}  {entry['question']} -> {entry['prediction']!r} (accepted: {entry['golds']})")
if report['verdict'] == 'not-measurable':
    print('No accepted answers exist for these questions, so nothing is scored; read the answers against the page yourself.')

## 8. Export outputs and provenance

Machine-readable JSON preserves every result (question, answer, span indices, score, window count), the evaluation report, the input manifest, the sample identity, digest, OCR source, words, boxes and accepted answers, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, and the runtime identity (Python, `torch`, `transformers`, device). The question/answer pairs are also written as CSV with explicit `image`, `question`, `answer`, `score`, `start`, `end` columns, and an annotated PNG draws the answer span's word boxes on the page (one colour per question) for visual inspection — a supplement to, not a replacement for, the machine-readable files. No credentials are recorded.

In [ ]:
import csv

COLOURS = [(200, 30, 30), (0, 140, 0), (40, 90, 220), (200, 120, 0), (130, 0, 160)]
annotated = image.convert('RGB').copy()
draw = ImageDraw.Draw(annotated)
for index, result in enumerate(results):
    if result['start'] is None:
        continue
    colour = COLOURS[index % len(COLOURS)]
    for box in boxes[result['start'] : result['end'] + 1]:
        draw.rectangle([box[0] - 2, box[1] - 2, box[2] + 2, box[3] + 2], outline=colour, width=2)
annotated.save('outputs/layoutlm_document_qa_annotated.png')
payload = {
    'predictions': results,
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'sample': {'kind': sample_kind, 'name': image_name, 'size': list(image.size), 'rgb_sha256': image_sha256, 'ocr_source': ocr_source, 'words': words, 'boxes': boxes, 'questions': questions, 'accepted_answers': golds},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
    },
}
with open('outputs/layoutlm_document_qa_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
with open('outputs/layoutlm_document_qa_answers.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['image', 'question', 'answer', 'score', 'start', 'end'])
    for result in results:
        writer.writerow([image_name, result['question'], result['answer'], f"{result['score']:.6f}", result['start'], result['end']])
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The answers are spans of the OCR words the model was given; the model never saw the pixels, so everything rests on the OCR — on the synthetic form the words and boxes come from the renderer and are perfect, which no real OCR is. The span score ranks spans within one page under the model's own head and is not calibrated: the smoke run's unanswerable question still got a span (`14 Harbour Road,` at 0.28), and a deployment must choose and validate its own rejection threshold on labelled pages. On the synthetic form the `anls` and `exact_match` values in the evaluation report compare the answers with values you rendered yourself and the verdict is `sample-sanity`, which proves only that the input contract, encoding, forward pass and span decoding work (the repository's smoke run scored 5/5 exact on this page); they say nothing about real OCR errors, scans, handwriting, non-Latin scripts, questions needing arithmetic or reasoning, answers that are not contiguous spans, or pages longer than one window, and a BYOD result is a single-page observation with the verdict `not-measurable`. **The model returns a span for every question**, and a page with more than 512 tokens is answered window by window with the best span across windows. The pipeline provides no OCR, no abstention, no multi-page handling, no benchmark evaluation and no training capability.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model, validate the demonstrated request, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** ask `What is the delivery address?` and compare its score with the five answerable ones; corrupt one OCR word (`words[12] = 'NW-2O26-O417'`) and watch the answer inherit the error; repeat the words nine times to exceed one window and read `n_windows`; enable `USE_BYOD` with a page you know and your own OCR JSON, then pass your own accepted answers to `evaluation_report` to see the verdict switch to `sample-sanity`.

## References

- Repository README: https://github.com/kurtvalcorza/layoutlm-document-qa-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/layoutlm-document-qa-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/layoutlm-document-qa-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/impira/layoutlm-document-qa
- Upstream architecture code (LayoutLM, Microsoft): https://github.com/microsoft/unilm/tree/master/layoutlm
- LayoutLM: Pre-training of Text and Layout for Document Image Understanding (Xu et al., 2019): https://arxiv.org/abs/1912.13318
- DocVQA: A Dataset for VQA on Document Images (Mathew, Karatzas, Jawahar, 2020): https://arxiv.org/abs/2007.00398
- Scene Text Visual Question Answering — the ANLS metric (Biten et al., 2019): https://arxiv.org/abs/1905.13648